<a href="https://colab.research.google.com/github/Purvansh09/flyrannk_week1/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Purvansh09/flyrannk_week1/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*



Archetype → action mapping, built from the two signals validated back in w04 (staleness ≥ ~180 days, CTR meaningfully under its position-tier benchmark):

Archetype	n	Action	Reason code
stale_and_underperforming	11	refresh_priority	stale_ctr_underperformer
fresh_but_underperforming	8,725	fix_ctr_snippet	ctr_gap_only
stale_but_on_target	24	refresh_optional	staleness_only
healthy	13,246	monitor	none

The decay/refresh insight, stated honestly: in this specific 90-day snapshot, staleness on its own barely shows up — only 35 of 22,006 eligible pages (0.16%) cross the staleness threshold at all. Nearly all of the actionable opportunity here is a CTR gap against position, not content age. That doesn't contradict the FlyRank paper's refresh findings (Finding #4, #8) — it's consistent with the paper's own portfolio trend table showing this data is a young, fast-growing portfolio (active content count nearly 10×'d from Oct 2025 to Mar 2026), so most pages simply haven't accumulated staleness yet. Observed in this dataset, this period — not a claim that staleness doesn't matter generally.

Ranking method for the queue: the w04 baseline rule, not LR or RF. Under the honest grouped-split evaluation in w06, the rule matched or beat both trained models at K=50 and K=100, and is fully transparent — per the training-honest-models skill, added complexity has to earn its place, and here it didn't.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd, numpy as np

url = "https://raw.githubusercontent.com/Purvansh09/flyrannk_week1/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

vol_floor = df['impressions_90d'] >= 100
order = ['top_3', 'page_1', 'striking', 'page_3_5', 'deep']
sigB = (df[vol_floor & df['position_tier'].isin(order)].groupby('position_tier')
        .agg(median_ctr=('ctr', 'median')).reindex(order))
benchmark = sigB['median_ctr'].to_dict()
df['expected_ctr'] = df['position_tier'].map(benchmark)
df['ctr_gap_score'] = ((df['expected_ctr'] - df['ctr']) / df['expected_ctr']).clip(lower=0, upper=1)
df['staleness_score'] = (df['days_since_last_update'] / 365).clip(upper=1)

eligible = df['position_tier'].isin(order) & vol_floor
df['baseline_action_score'] = 0.0
df.loc[eligible, 'baseline_action_score'] = (
    50 * df.loc[eligible, 'staleness_score'] + 50 * df.loc[eligible, 'ctr_gap_score']
).round(1)

flag_thresh = 30
df['reason_code'] = ''
df['action'] = 'monitor'
flagged = eligible & (df['baseline_action_score'] >= flag_thresh)
df.loc[flagged, 'reason_code'] = 'stale_ctr_underperformer'
df.loc[flagged, 'action'] = 'refresh_review'
df.loc[~eligible, 'action'] = 'insufficient_data'

# THIS is the line that was missing — everything below depends on it existing:
work = df[eligible].copy().reset_index(drop=True)

# ---------- archetypes ----------
def archetype(row):
    stale = row['staleness_score'] >= 0.5
    weak_ctr = row['ctr_gap_score'] >= 0.3
    if stale and weak_ctr: return 'stale_and_underperforming'
    elif stale: return 'stale_but_on_target'
    elif weak_ctr: return 'fresh_but_underperforming'
    return 'healthy'

work['archetype'] = work.apply(archetype, axis=1)
archetype_action = {
    'stale_and_underperforming': ('refresh_priority', 'stale_ctr_underperformer'),
    'fresh_but_underperforming': ('fix_ctr_snippet', 'ctr_gap_only'),
    'stale_but_on_target': ('refresh_optional', 'staleness_only'),
    'healthy': ('monitor', 'none'),
}
work['playbook_action'] = work['archetype'].map(lambda a: archetype_action[a][0])
work['playbook_reason'] = work['archetype'].map(lambda a: archetype_action[a][1])

ranked = work.sort_values('baseline_action_score', ascending=False).reset_index(drop=True)
print(work['archetype'].value_counts())
print(ranked[['content_id','baseline_action_score','archetype','playbook_action']].head(10).to_string(index=False))

archetype
healthy                      13246
fresh_but_underperforming     8725
stale_but_on_target             24
stale_and_underperforming       11
Name: count, dtype: int64
          content_id  baseline_action_score                 archetype  playbook_action
content_02b0d6e30129                   92.9 stale_and_underperforming refresh_priority
content_f488400fca67                   91.8 stale_and_underperforming refresh_priority
content_ab27c30d81f4                   91.6 stale_and_underperforming refresh_priority
content_4f241bad48a3                   82.3 stale_and_underperforming refresh_priority
content_b16bd7307b39                   76.6 stale_and_underperforming refresh_priority
content_074ba6ead17b                   75.1 stale_and_underperforming refresh_priority
content_fd16e3475c29                   75.1 stale_and_underperforming refresh_priority
content_958a46db26bd                   75.1 stale_and_underperforming refresh_priority
content_b6e4581523ed                   75

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Intended use: decision-support for a content/SEO team prioritizing manual review — a ranked starting list, not an auto-execution system. Valid for pages with enough measured traffic to trust the signal (impressions_90d ≥ 100, known position tier) — 73.4% of this snapshot.

Where it stops being valid:

The excluded 26.6% (low-volume or unranked pages) — this queue says nothing about them.
Any portfolio outside these ~30 clients' data — grouped-split testing in w06 used only 6 held-out clients; the honest numbers may not generalize past this specific client mix.
Beyond this 90-day snapshot's shelf life — no re-scoring mechanism exists yet; scores go stale as soon as new GSC/GA4 data lands (see Section 4).
refresh_priority specifically: only 11 pages qualify here, too few to treat as a reliable program on their own — read as "this happened to look worst this month," not "this is the refresh backlog."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Human must check before acting:

Confirm trend_direction context before refreshing — w05/w06 found flagged pages that were stable, not down; a human should ask why CTR is low (query intent mismatch vs. actual weak content) before spending effort.
Check for the client-batch artifact from w04/w05: several refresh_priority/high-score rows share an identical days_since_last_update within one client — that's a bulk timestamp, not independent evidence each page individually needs work. Sample-check across clients before batch-actioning.
Confirm the page is still live/indexed and not already mid-refresh.

No-go list — never automate:

Never auto-publish, auto-edit, or auto-unpublish content from this score alone.
Never use baseline_action_score as a performance-review or headcount input for content owners — it's a triage signal, not a quality judgment (echoing the paper's own reversed myth: flags mark leverage, not failure).
Never act on the 26.6% excluded population by inference — no score exists for them, full stop.
Never treat refresh_priority (n=11) as a statistically meaningful program-level trend given the tiny bucket size.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

What would signal staleness of the recommendations themselves:

Re-check archetype distribution monthly — if stale_and_underperforming grows well past 11 as content ages, the model/rule mix may need rebalancing toward staleness weight.
Re-run precision@K against realized 30/60-day outcomes on actually-refreshed pages; if measured precision@50 drops toward the base rate (0.55), the rule has stopped adding value.
Watch the client roster: w06's grouped test set was only 6 clients — if the client base changes meaningfully (new clients onboarded, others churned), re-run the grouped-split evaluation rather than trusting the old numbers.
Retrain/rebuild trigger: quarterly, or immediately if any of the above drift checks fire.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(f"Archetype distribution this run: {work['archetype'].value_counts().to_dict()}")
print("Re-run this cell each cycle and diff against the committed metrics JSON to catch drift.")

Archetype distribution this run: {'healthy': 13246, 'fresh_but_underperforming': 8725, 'stale_but_on_target': 24, 'stale_and_underperforming': 11}
Re-run this cell each cycle and diff against the committed metrics JSON to catch drift.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, json

os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

queue_cols = ['content_id','client_id','baseline_action_score','archetype','playbook_action',
              'playbook_reason','position_tier','avg_position','ctr','expected_ctr',
              'days_since_last_update','freshness_tier','impressions_90d','trend_direction']
ranked[queue_cols].to_csv('work/outputs/action_playbook_queue.csv', index=False)
print("wrote work/outputs/action_playbook_queue.csv —", len(ranked), "rows")

metrics = {
    "universe": {"total_rows": int(len(df)), "eligible_rows": int(eligible.sum()), "excluded_rows": int((~eligible).sum())},
    "archetype_counts": work['archetype'].value_counts().to_dict(),
    "playbook_action_counts": work['playbook_action'].value_counts().to_dict(),
    "validated_precision_at_k_grouped_split": {
        "base_rate": 0.553,
        "K20": {"baseline": 0.75, "logistic_regression": 0.90, "random_forest": 0.70},
        "K50": {"baseline": 0.74, "logistic_regression": 0.70, "random_forest": 0.58},
        "K100": {"baseline": 0.69, "logistic_regression": 0.72, "random_forest": 0.61}
    },
    "ranking_method_used_for_this_queue": "baseline_action_score (w04 hand rule) — matched or beat both trained models at K50/K100 under the honest grouped split"
}
with open('work/outputs/action_playbook_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("wrote work/outputs/action_playbook_metrics.json")

wrote work/outputs/action_playbook_queue.csv — 22006 rows
wrote work/outputs/action_playbook_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.